<div style="border-top:4px solid #0f766e;padding:28px 0 18px"><div style="color:#0f766e;font-weight:700;letter-spacing:.8px">MODULE 12: PUBLISHING, PERMISSIONS, AND AUDIT</div><div style="color:#17212b;font-size:30px;font-weight:750">Publishing, permissions, and audit</div><p style="color:#475569;line-height:1.7">Run in order against a dedicated Level 3 course database. Results are rendered as tables and all examples are scoped to this module.</p></div>

## Boundary

Do not grant access to, truncate, or alter a shared production object from this lab. Review every object name before executing a write.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab

lab = WarehouseLab()


In [ ]:
lab.sql("SELECT CURRENT_USER() AS current_user, DATABASE() AS current_database", title="Release operator identity")
lab.sql("SHOW GRANTS", title="Existing grants before the release")

In [ ]:
lab.execute("CREATE ROLE IF NOT EXISTS course_bi_reader_l3")
lab.execute(f"GRANT SELECT_PRIV ON internal.{lab.database}.orders_imported TO ROLE 'course_bi_reader_l3'")
lab.sql("SHOW GRANTS FOR ROLE 'course_bi_reader_l3'", title="BI reader role grants")
lab.sql("""
SELECT 'course_bi_reader_l3' AS role_name,
       'orders_imported' AS object_name,
       'SELECT_PRIV' AS privilege,
       'approved release' AS audit_reason
""", title="Release audit record")

In [ ]:
release_runbook = """
-- CREATE USER 'course_bi_demo' IDENTIFIED BY '<secret-from-the-secret-manager>';
-- GRANT course_bi_reader_l3 TO 'course_bi_demo';
-- REVOKE course_bi_reader_l3 FROM 'course_bi_demo';
"""
print(release_runbook)
lab.sql("SHOW GRANTS FOR ROLE 'course_bi_reader_l3'", title="Final privilege review", final=True)

## Takeaway

Operational correctness includes scope, evidence, reversibility, and ownership. Record what changed and how the result was checked before declaring the exercise complete.